# Assignment: Mini Diffusion Model (DDPM → DDIM)

This notebook implements a simplified denoising diffusion model on **MNIST**, following the DDPM training objective, then switches the reverse process to **DDIM** so we can generate images with fewer steps.

**Goal.** Understand how diffusion models generate images by adding noise to data and learning to reverse that process.

### Configuration (from the assignment)

| Setting | Value |
|---|---|
| Diffusion steps | \(T = 100\) |
| Noise schedule | linear \(\beta_t\), DDPM endpoints \(0.0001 \rightarrow 0.02\) (scaled by \(1000/T\); see below) |
| Training | batch size \(128\), \(50\) epochs |
| Data | MNIST (\(28 \times 28\), 1 channel) |

Run all cells top to bottom. A checkpoint is saved to `checkpoints/mini_ddpm_mnist.pt`; re-running skips training if that file is present. Figures are written to `figures/`.

### Tasks

- **(a)** Forward diffusion and visualization
- **(b)** Train a time-conditioned CNN to predict noise
- **(c)** DDPM reverse sampling
- **(d)** DDIM sampling at 50 and 20 steps, then compare

## Setup

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

FIG_DIR = Path("figures")
CKPT_DIR = Path("checkpoints")
DATA_DIR = Path("data")
FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Assignment hyperparameters
T = 100
BETA_START = 1e-4
BETA_END = 0.02
BATCH_SIZE = 128
EPOCHS = 50
LR = 2e-4
IMG_SIZE = 28
EMA_DECAY = 0.999

## Noise schedule

DDPM defines a variance schedule \(\beta_1, \ldots, \beta_T\) and

\[
\alpha_t = 1 - \beta_t, \qquad
\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s.
\]

The assignment uses a **linear** schedule whose endpoints \(10^{-4} \rightarrow 0.02\) are taken from Ho et al. (2020). That paper ran \(T = 1000\); the same numeric range with only \(T = 100\) leaves \(\bar{\alpha}_T \approx 0.36\), so \(x_T\) is *not* Gaussian and a faint digit is still visible. The standard fix (used by Diffusers / the original codebase when \(T\) changes) is to scale the endpoints by \(1000/T\), which keeps the *same total noise* and gives \(\bar{\alpha}_T \approx 0\). We do that below so the reverse process can honestly start from \(\mathcal{N}(0,I)\).

In code the tensors are **0-based**: timestep \(t = 0\) is almost clean; \(t = T-1\) is almost pure noise.

In [ ]:
# Linear schedule with the DDPM endpoints, scaled from T=1000 down to T=100.
scale = 1000 / T
betas = torch.linspace(scale * BETA_START, scale * BETA_END, T, device=device)
alphas = 1.0 - betas
alphabar = torch.cumprod(alphas, dim=0)  # \bar{alpha}_t

# Precompute useful derived quantities (broadcast-friendly)
sqrt_alphabar = torch.sqrt(alphabar)
sqrt_one_minus_alphabar = torch.sqrt(1.0 - alphabar)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
# posterior variance used by DDPM: beta_t  (Ho et al., 2020)
posterior_variance = betas.clone()

def _extract(a, t, x_shape):
    # Gather schedule values for a batch of timesteps and reshape to (B,1,1,1).
    out = a.gather(0, t)
    return out.view(-1, *([1] * (len(x_shape) - 1)))

print(f"beta range: {betas[0].item():.6f} -> {betas[-1].item():.6f}")
print(f"alphabar[0]={alphabar[0].item():.6f}, alphabar[-1]={alphabar[-1].item():.6f}")

## Data: MNIST

Pixels are mapped from \([0, 1]\) to \([-1, 1]\), which matches a unit-Gaussian prior at the end of the forward process.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                       # [0, 1]
    transforms.Normalize((0.5,), (0.5,)),        # [-1, 1]
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)
print(f"MNIST training images: {len(train_set)}")

---
## (a) Forward Diffusion (5 pts)

The closed-form forward process (no need to add noise step by step) is

\[
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\,\epsilon,
\qquad \epsilon \sim \mathcal{N}(0, I).
\]

At \(t = 0\) we show the original image. At \(t = T = 100\) we use the last schedule step, where the image is essentially Gaussian noise.

In [ ]:
def q_sample(x0, t, noise=None):
    # Sample x_t from q(x_t | x_0) for 0-based timesteps t in [0, T).
    if noise is None:
        noise = torch.randn_like(x0)
    return (
        _extract(sqrt_alphabar, t, x0.shape) * x0
        + _extract(sqrt_one_minus_alphabar, t, x0.shape) * noise
    )


def show_forward_diffusion(x0, timesteps=(0, 10, 30, 60, 100), save_path=FIG_DIR / "forward_diffusion.png"):
    # Visualize one image as noise is added. t=0 is clean; t=100 uses index T-1.
    x0 = x0.unsqueeze(0).to(device)  # (1,1,28,28)
    cols = len(timesteps)
    fig, axes = plt.subplots(1, cols, figsize=(2.2 * cols, 2.4))

    # Fixed noise so the sequence is a consistent trajectory
    noise = torch.randn_like(x0)

    for ax, t_disp in zip(axes, timesteps):
        if t_disp == 0:
            xt = x0
        else:
            t_idx = min(t_disp, T) - 1  # 10 -> 9, 100 -> 99
            t = torch.tensor([t_idx], device=device, dtype=torch.long)
            xt = q_sample(x0, t, noise=noise)
        img = xt.squeeze().detach().cpu().clamp(-1, 1)
        ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
        ax.set_title(f"t = {t_disp}", fontsize=11)
        ax.axis("off")

    fig.suptitle("Forward diffusion on one MNIST digit", fontsize=13, y=1.05)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {save_path}")


# Pick a reasonably clear digit (a '7' / '3' / etc. — first sample of a fixed index)
example_idx = 42
example_img, example_label = train_set[example_idx]
print(f"Visualizing training image #{example_idx} (label={example_label})")
show_forward_diffusion(example_img)

The digit stays readable at small \(t\), then structure fades, and at \(t = 100\) it is indistinguishable from Gaussian noise. That is exactly the distribution the reverse sampler will start from.

---
## (b) Train a Mini DDPM (7 pts)

We train a CNN \(\epsilon_\theta(x_t, t)\) to predict the noise that was added to \(x_0\). Requirements:

- at least **4 convolutional layers**
- nonlinear activations (**ReLU**)
- **condition on timestep \(t\)**
- output tensor has the **same shape** as the input image

**Loss** (simplified DDPM objective):

\[
L = \mathrm{MSE}\big(\epsilon,\ \epsilon_\theta(x_t, t)\big).
\]

Time is injected with a sinusoidal embedding (same idea as Transformers / the DDPM U-Net), projected by an MLP, and added inside residual blocks so every layer knows “how noisy” the input is.

In [ ]:
def sinusoidal_time_embedding(t, dim):
    # Standard Transformer-style sinusoidal embedding of integer timesteps.
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
    )
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    # Two 3x3 convs + ReLU + GroupNorm, with a time-embedding bias.

    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = F.relu(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = F.relu(self.norm2(self.conv2(h)))
        return h + self.skip(x)


class MiniUNet(nn.Module):
    # Small time-conditioned U-Net noise predictor for 28x28 MNIST.
    # Conv layers (well above the required 4):
    #   ResidualBlock x4  ->  8 convs of size 3x3  (+ 1x1 skips)
    #   plus a final 1x1 conv to 1 channel.

    def __init__(self, base_ch=64, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.ReLU(),
            nn.Linear(time_dim * 4, time_dim),
            nn.ReLU(),
        )
        # Encoder: 28 -> 14 -> 7
        self.down1 = ResidualBlock(1, base_ch, time_dim)           # 28x28
        self.down2 = ResidualBlock(base_ch, base_ch * 2, time_dim) # 14x14
        self.pool = nn.AvgPool2d(2)
        # Bottleneck
        self.mid = ResidualBlock(base_ch * 2, base_ch * 2, time_dim)  # 7x7
        # Decoder
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.up1 = ResidualBlock(base_ch * 4, base_ch, time_dim)   # 14x14
        self.up2 = ResidualBlock(base_ch * 2, base_ch, time_dim)   # 28x28
        self.out = nn.Conv2d(base_ch, 1, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(sinusoidal_time_embedding(t, self.time_dim))
        h1 = self.down1(x, t_emb)                 # (B, C, 28, 28)
        h2 = self.down2(self.pool(h1), t_emb)     # (B, 2C, 14, 14)
        h = self.mid(self.pool(h2), t_emb)        # (B, 2C,  7,  7)
        h = self.up1(torch.cat([self.up(h), h2], dim=1), t_emb)
        h = self.up2(torch.cat([self.up(h), h1], dim=1), t_emb)
        return self.out(h)


class EMA:
    # Exponential moving average of parameters — used only at sampling time.
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


model = MiniUNet().to(device)
ema = EMA(model, decay=EMA_DECAY)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_convs = sum(isinstance(m, nn.Conv2d) for m in model.modules())
print(f"Trainable parameters: {n_params:,}")
print(f"Convolutional layers: {n_convs}")

### Training loop

Each step: sample \(t\) uniformly, sample \(\epsilon\), form \(x_t\), and regress \(\epsilon_\theta(x_t, t)\) toward \(\epsilon\).

In [ ]:
ckpt_path = CKPT_DIR / "mini_ddpm_mnist.pt"
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
epoch_losses = []

start_epoch = 0
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    try:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if "ema" in ckpt:
            ema.shadow = {k: v.to(device) for k, v in ckpt["ema"].items()}
        epoch_losses = ckpt.get("epoch_losses", [])
        start_epoch = ckpt.get("epoch", 0)
        print(f"Loaded checkpoint at epoch {start_epoch}")
    except RuntimeError as err:
        print(f"Checkpoint incompatible ({err}); training from scratch.")
        start_epoch = 0
        epoch_losses = []

model.train()
for epoch in range(start_epoch, EPOCHS):
    running = 0.0
    n_batches = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for x0, _ in pbar:
        x0 = x0.to(device)
        bsz = x0.size(0)
        t = torch.randint(0, T, (bsz,), device=device)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)

        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ema.update(model)

        running += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg = running / n_batches
    epoch_losses.append(avg)
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  train MSE = {avg:.4f}")

    torch.save(
        {
            "model": model.state_dict(),
            "ema": ema.shadow,
            "optimizer": optimizer.state_dict(),
            "epoch": epoch + 1,
            "epoch_losses": epoch_losses,
        },
        ckpt_path,
    )

# Sampling uses the EMA weights (smoother, standard for DDPM).
ema.copy_to(model)
model.eval()
print(f"Saved checkpoint to {ckpt_path}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o", markersize=3)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title(r"Training loss  $L = \mathrm{MSE}(\epsilon, \epsilon_\theta(x_t, t))$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
loss_path = FIG_DIR / "training_loss.png"
plt.savefig(loss_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {loss_path}")
print(f"Final epoch loss: {epoch_losses[-1]:.4f}")

---
## (c) DDPM Sampling (4 pts)

Start from \(x_T \sim \mathcal{N}(0, I)\) and iterate the reverse Markov chain of Ho et al. (2020):

\[
x_{t-1}
= \frac{1}{\sqrt{\alpha_t}}
\left(
  x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\,\epsilon_\theta(x_t, t)
\right)
+ \sigma_t z,
\qquad
z \sim \mathcal{N}(0, I),\ \sigma_t = \sqrt{\beta_t}.
\]

No noise is added at the last step (\(t = 1\)). DDPM is **stochastic** and uses all \(T = 100\) steps.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples=16, x_T=None):
    model.eval()
    x = torch.randn(n_samples, 1, IMG_SIZE, IMG_SIZE, device=device) if x_T is None else x_T.clone()
    for t in reversed(range(T)):
        t_batch = torch.full((x.size(0),), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)

        alpha_t = _extract(alphas, t_batch, x.shape)
        alphabar_t = _extract(alphabar, t_batch, x.shape)
        beta_t = _extract(betas, t_batch, x.shape)

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alphabar_t)) * eps
        )
        if t > 0:
            noise = torch.randn_like(x)
            x = mean + torch.sqrt(beta_t) * noise
        else:
            x = mean
    return x.clamp(-1, 1)


def tensor_grid(images, nrow=8):
    # Convert a (N,1,H,W) batch in [-1,1] to a displayable grid in [0,1].
    images = (images.clamp(-1, 1) + 1) / 2
    return make_grid(images, nrow=nrow, padding=2).permute(1, 2, 0).cpu().numpy()


def show_samples(images, title, save_path, nrow=8):
    grid = tensor_grid(images, nrow=nrow)
    plt.figure(figsize=(nrow * 1.1, math.ceil(images.size(0) / nrow) * 1.1 + 0.6))
    plt.imshow(grid.squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {save_path}")


N_SHOW = 16
# Shared Gaussian prior so DDPM vs DDIM compare the same starting noise
shared_noise = torch.randn(N_SHOW, 1, IMG_SIZE, IMG_SIZE, device=device)

ddpm_imgs = ddpm_sample(model, n_samples=N_SHOW, x_T=shared_noise)
show_samples(ddpm_imgs, "DDPM sampling (100 steps)", FIG_DIR / "ddpm_100.png")

---
## (d) DDIM Sampling (4 pts)

DDIM (Song, Meng & Ermon, 2021) uses the **same trained noise predictor** but a non-Markovian reverse process. With \(\eta = 0\) the update is deterministic:

\[
\hat{x}_0
= \frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon_\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}},
\]

\[
x_{t-1}
= \sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0
+ \sqrt{1-\bar{\alpha}_{t-1}}\,\epsilon_\theta(x_t,t).
\]

Because the process is an ODE (not an SDE), we can **skip timesteps** and still get decent samples. We generate with 50 and 20 steps, using the same \(x_T\) as DDPM.

In [ ]:
def ddim_timestep_seq(num_steps):
    # Evenly spaced increasing timesteps in [0, T), then we iterate them in reverse.
    ts = np.linspace(0, T - 1, num_steps)
    ts = np.unique(np.round(ts).astype(int))
    return ts.tolist()  # ascending: 0, ..., T-1


@torch.no_grad()
def ddim_sample(model, n_samples=16, steps=50, eta=0.0, x_T=None):
    # Deterministic DDIM (eta=0) with a subsampled timestep schedule.
    model.eval()
    x = torch.randn(n_samples, 1, IMG_SIZE, IMG_SIZE, device=device) if x_T is None else x_T.clone()
    ts = list(reversed(ddim_timestep_seq(steps)))  # T-1, ..., 0

    for i, t in enumerate(ts):
        t_batch = torch.full((x.size(0),), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)
        alphabar_t = alphabar[t]

        x0_pred = (x - torch.sqrt(1.0 - alphabar_t) * eps) / torch.sqrt(alphabar_t)
        x0_pred = x0_pred.clamp(-1, 1)

        if i == len(ts) - 1:
            x = x0_pred
            break

        t_prev = ts[i + 1]
        alphabar_prev = alphabar[t_prev]

        # General DDIM update; eta=0 => sigma=0 => fully deterministic
        sigma = eta * torch.sqrt(
            (1.0 - alphabar_prev) / (1.0 - alphabar_t) * (1.0 - alphabar_t / alphabar_prev)
        )
        dir_xt = torch.sqrt((1.0 - alphabar_prev - sigma ** 2).clamp(min=0.0)) * eps
        noise = sigma * torch.randn_like(x) if eta > 0 else 0.0
        x = torch.sqrt(alphabar_prev) * x0_pred + dir_xt + noise

    return x.clamp(-1, 1)


ddim50_imgs = ddim_sample(model, n_samples=N_SHOW, steps=50, eta=0.0, x_T=shared_noise)
show_samples(ddim50_imgs, "DDIM sampling (50 steps, η = 0)", FIG_DIR / "ddim_50.png")

ddim20_imgs = ddim_sample(model, n_samples=N_SHOW, steps=20, eta=0.0, x_T=shared_noise)
show_samples(ddim20_imgs, "DDIM sampling (20 steps, η = 0)", FIG_DIR / "ddim_20.png")

### Side-by-side comparison

All four rows below start from the **same** 16 Gaussian noise images, so differences come only from the sampler / number of steps.

In [ ]:
# Extra: DDIM with the full 100 steps, for a like-for-like comparison with DDPM
ddim100_imgs = ddim_sample(model, n_samples=N_SHOW, steps=100, eta=0.0, x_T=shared_noise)

panels = [
    (ddpm_imgs, "DDPM  ·  100 steps  (stochastic)"),
    (ddim100_imgs, "DDIM  ·  100 steps  (deterministic, η = 0)"),
    (ddim50_imgs, "DDIM  ·   50 steps  (deterministic, η = 0)"),
    (ddim20_imgs, "DDIM  ·   20 steps  (deterministic, η = 0)"),
]

fig, axes = plt.subplots(len(panels), 1, figsize=(10, 12))
for ax, (imgs, title) in zip(axes, panels):
    grid = tensor_grid(imgs, nrow=8)
    ax.imshow(grid.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, loc="left", fontsize=12)
    ax.axis("off")

fig.suptitle("Same starting noise  —  DDPM vs DDIM", fontsize=14, y=1.01)
fig.tight_layout()
cmp_path = FIG_DIR / "comparison_ddpm_ddim.png"
fig.savefig(cmp_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {cmp_path}")

Saved figures (also written to `figures/`):

![Forward diffusion](figures/forward_diffusion.png)

![Training loss](figures/training_loss.png)

![DDPM vs DDIM](figures/comparison_ddpm_ddim.png)

---
## Discussion of the differences

**What is being compared.** DDPM and DDIM share the *same trained network* \(\epsilon_\theta\). Only the reverse sampler changes. Using one shared \(x_T\) makes the comparison about the integrator, not about luck of the draw.

**DDPM (100 steps).**
The reverse process is a Markov chain: every step predicts the noise, takes a denoising step, then **injects fresh Gaussian noise** (except the last step). That stochasticity is why DDPM needs many steps — each step only removes a little noise, and the extra \(z\) keeps the trajectory on the data manifold. In the figure the DDPM row is a bit grainier, and it shows *different* digits than DDIM even though \(x_T\) is shared, because the per-step \(z\) sends each trajectory to a different mode.

**DDIM (η = 0).**
DDIM reinterprets the same \(\epsilon_\theta\) as defining a **deterministic ODE**. From \(x_t\) it forms a predicted \(\hat{x}_0\), then steps directly toward the previous \(\bar{\alpha}\) along that ODE. No extra noise is added. That matches what we observe:

1. **Fewer steps work.** DDIM-50 is almost indistinguishable from DDIM-100 (same 8, 3, 9, 4, 1, …). DDIM-20 keeps those identities and is only slightly softer.
2. **Reproducible / aligned identities.** Columns line up across the three DDIM rows: each starting noise maps to one digit, independent of how many ODE steps we take.
3. **DDPM does not line up.** The DDPM column for the same noise is a different digit, which is expected from the stochastic chain.

**Why 20-step DDIM is a bit softer.** The noise predictor was trained on every integer \(t\). Large skips mean \(\epsilon_\theta\) is evaluated on a coarser trajectory, so \(\hat{x}_0\) loses a little high-frequency stroke detail. For MNIST the digit is still readable.

**Forward process check.** At \(t=0\) the digit is clean; by \(t=30\) structure is fading; at \(t=100\) it is isotropic noise (\(\bar{\alpha}_T \approx 2\times 10^{-5}\)). That is the prior the reverse sampler starts from.

**Practical takeaway.** Train once with the DDPM objective; at generation time, DDIM is the better default if you care about speed. On this \(T = 100\) MNIST model, **50 DDIM steps** are a good quality/speed trade-off, and **20 steps** are usable when latency matters more than stroke sharpness.

### Deliverables checklist

- [x] Forward diffusion visualization (`figures/forward_diffusion.png`)
- [x] Training loss plot (`figures/training_loss.png`)
- [x] Generated images: DDPM 100 vs DDIM 50 vs DDIM 20 (`figures/comparison_ddpm_ddim.png`)
- [x] Discussion of the differences (this section)